In [ ]:
#Makes data set with all potential loan-month observations. Used for out-of-sample cumulative prepayment prediction.
# #Read reduced data set and do further cleaning and variable transformations
import pandas as pd
import numpy as np
import numpy_financial as nf
from bisect import bisect_right
import gc
from pathlib import Path

data_path2 = Path.home() / "Prepayment_Analysis_Data" / "mortgage_data_test.csv"
KEEP_COLUMNS = ["LOAN_ID", "ORIG_RATE",  "ORIG_UPB", "ORIG_TERM", "LOAN_AGE","OLTV", "CSCORE_B", "CSCORE_C", "PROP","STATE", "MSA", "ORIG_DATE"]
col_type_=['string'] + ['float32'] * 7 + ['string'] * 4
mort_clean_df=pd.read_csv(data_path2, header=0, dtype=dict(zip(KEEP_COLUMNS, col_type_)))
mort_clean_df.dtypes
mort_clean_df = mort_clean_df.query("ORIG_UPB >= 10000 and PROP!='MH' and ORIG_TERM==360 and STATE not in ['PR', 'VI', 'GU', 'MP', 'AS']")
mort_clean_df['CSCORE_MAX'] = mort_clean_df[["CSCORE_B", "CSCORE_C"]].max(axis=1)
mort_clean_df = mort_clean_df[~mort_clean_df['CSCORE_MAX'].isna()]
mort_clean_df['coop_condo_dummy'] = mort_clean_df['PROP'].apply(lambda x: 1 if x in ['CO', 'CP'] else 0)
mort_clean_df.drop(columns=["CSCORE_B", "CSCORE_C", "PROP", "ORIG_TERM"], inplace=True)
#I can make the code more efficient calculating orig_date to orig_date1 in a separate df and merging back
mort_clean_df['ORIG_MONTH'] = mort_clean_df['ORIG_DATE'].str[:2].astype('int32')
mort_clean_df['ORIG_YEAR'] = mort_clean_df['ORIG_DATE'].str[2:].astype('int32')
mort_clean_df['ORIG_DATE1'] = pd.to_datetime(dict(year=mort_clean_df['ORIG_YEAR'], month=mort_clean_df['ORIG_MONTH'], day=1))
mort_clean_df.drop(columns=["ORIG_DATE", "ORIG_MONTH", "ORIG_YEAR"], inplace=True)
mort_clean_df.describe()

,ORIG_RATE,ORIG_UPB,LOAN_AGE,OLTV,DLQ_STATUS,CSCORE_MAX,coop_condo_dummy,ORIG_DATE1
count,5.687635e+06,5.687635e+06,5.652231e+06,5.687635e+06,5.687635e+06,5.687635e+06,5.687635e+06,5687635
mean,6.353510e+00,3.227369e+05,1.498656e+01,7.716487e+01,4.533818e-02,7.617906e+02,9.968994e-02,2022-12-29 18:42:58.065433088
min,2.750000e+00,1.500000e+04,-1.000000e+00,3.000000e+00,0.000000e+00,6.200000e+02,0.000000e+00,2020-12-01 00:00:00
25%,5.990000e+00,1.930000e+05,7.000000e+00,7.000000e+01,0.000000e+00,7.380000e+02,0.000000e+00,2022-12-01 00:00:00
50%,6.375000e+00,2.900000e+05,1.500000e+01,8.000000e+01,0.000000e+00,7.700000e+02,0.000000e+00,2023-01-01 00:00:00
75%,6.750000e+00,4.210000e+05,2.300000e+01,9.400000e+01,0.000000e+00,7.930000e+02,0.000000e+00,2023-02-01 00:00:00
max,8.625000e+00,2.095000e+06,4.200000e+01,9.700000e+01,3.100000e+01,8.340000e+02,1.000000e+00,2023-03-01 00:00:00
std,6.567555e-01,1.734503e+05,9.235882e+00,1.825712e+01,5.602489e-01,3.944548e+01,2.995862e-01,NaN


In [2]:
#Feature engineering - add first default age, first modification and/or forbearance age, prepayment flag to main dataframe

#Create arliest defaulted date, where relevant, by loan
def_df= mort_clean_df[['LOAN_ID', 'DLQ_STATUS', 'LOAN_AGE']].copy()
def_df= def_df[~def_df["DLQ_STATUS"].isin([0, 1, 2])]
def_date_df = def_df.groupby('LOAN_ID')['LOAN_AGE'].min().reset_index()
def_date_df = def_date_df.rename(columns={'LOAN_AGE': 'FIRST_DEFAULTER_AGE'})
def_date_df=def_date_df[['LOAN_ID', 'FIRST_DEFAULTER_AGE']]


#earliest forbearance or modified date, where relevant, by loan
fb_mod_df= mort_clean_df[['LOAN_ID', 'MOD_FLAG', 'FORBEARANCE_INDICATOR', 'LOAN_AGE']].copy()
fb_mod_df= fb_mod_df[(mort_clean_df["MOD_FLAG"].isin(['Y'])) | (mort_clean_df["FORBEARANCE_INDICATOR"].isin(['F', 'T', 'R']))]
fb_mod_date_df = fb_mod_df.groupby('LOAN_ID')['LOAN_AGE'].min().reset_index()
fb_mod_date_df = fb_mod_date_df.rename(columns={'LOAN_AGE': 'FIRST_FB_MOD_AGE'})
fb_mod_date_df=fb_mod_date_df[['LOAN_ID', 'FIRST_FB_MOD_AGE']]

#last date loan observed. This will either be when the loan is prepaid, defaulted, modified/forborne, or the last observation in the dataset
last_mort_df = mort_clean_df[['LOAN_ID', 'LOAN_AGE']].copy().groupby('LOAN_ID')['LOAN_AGE'].max().reset_index()
last_mort_df = last_mort_df.rename(columns={'LOAN_AGE': 'LAST_MORT_AGE'})
last_mort_df=last_mort_df[['LOAN_ID', 'LAST_MORT_AGE']]

#merge first default, first forbearance/mod, and last observation data into main dataset
mort_clean_df= mort_clean_df.merge(def_date_df, on='LOAN_ID', how='left')
mort_clean_df= mort_clean_df.merge(fb_mod_date_df, on='LOAN_ID', how='left')
mort_clean_df= mort_clean_df.merge(last_mort_df, on='LOAN_ID', how='left')
mort_clean_df['FIRST_DEFAULTER_AGE']= mort_clean_df['FIRST_DEFAULTER_AGE'].fillna(999)   
mort_clean_df['FIRST_FB_MOD_AGE']= mort_clean_df['FIRST_FB_MOD_AGE'].fillna(999)   
mort_clean_df = mort_clean_df[mort_clean_df['LOAN_AGE'] <= mort_clean_df['FIRST_DEFAULTER_AGE'] ]
mort_clean_df = mort_clean_df[mort_clean_df['LOAN_AGE'] <= mort_clean_df['FIRST_FB_MOD_AGE'] ]

#Create default and forbearance/mod flags
mort_clean_df['def_flag'] = np.where(mort_clean_df['LOAN_AGE']== mort_clean_df['FIRST_DEFAULTER_AGE'], 1, 0)
mort_clean_df['fb_mod_flag'] = np.where(mort_clean_df['LOAN_AGE']== mort_clean_df['FIRST_FB_MOD_AGE'], 1, 0)
def_mod_fb_df = mort_clean_df[['LOAN_AGE', 'def_flag', 'fb_mod_flag']].groupby("LOAN_AGE").mean().reset_index()

#Create prepayment flag
#Assume loans that terminated via default or forbearance/modification do not prepay
#Both groups are tiny, can reintegrate them into the prepayment analysis eventually
mort_clean_df = mort_clean_df[(mort_clean_df['LOAN_AGE'] <= mort_clean_df['FIRST_DEFAULTER_AGE']) & (mort_clean_df['LOAN_AGE'] <= mort_clean_df['FIRST_FB_MOD_AGE'])]
mort_clean_df['PREPAY_FLAG'] = np.where(
  ( mort_clean_df['LOAN_AGE']== mort_clean_df['LAST_MORT_AGE'])
  & ( mort_clean_df['fb_mod_flag'] == 0)
  & ( mort_clean_df['def_flag'] == 0)
  & (mort_clean_df['LOAN_AGE'] < 360)
   , 1, 0)
mort_clean_df['QTR_ORIG'] = mort_clean_df['ORIG_DATE1'].dt.to_period('Q')
mort_clean_df['QTR_NOW'] = mort_clean_df['QTR_ORIG'] + (mort_clean_df['LOAN_AGE'] // 3).astype(int)
mort_clean_df.drop(columns=['FIRST_DEFAULTER_AGE', 'FIRST_FB_MOD_AGE', 'FORBEARANCE_INDICATOR', 'DLQ_STATUS', 'LAST_MORT_AGE'], inplace=True)
mort_clean_df = mort_clean_df[mort_clean_df['QTR_NOW'] <= pd.Period('2025Q1', freq='Q')]

In [3]:
actual_prepay_df=mort_clean_df[['LOAN_ID', 'LOAN_AGE', 'PREPAY_FLAG']].copy()
actual_prepay_df.describe()

,LOAN_AGE,PREPAY_FLAG
count,4.984788e+06,4.984788e+06
mean,1.313875e+01,6.230155e-03
std,8.145634e+00,7.868508e-02
min,-1.000000e+00,0.000000e+00
25%,6.000000e+00,0.000000e+00
50%,1.300000e+01,0.000000e+00
75%,2.000000e+01,0.000000e+00
max,3.800000e+01,1.000000e+00


In [4]:
#market rate data
data_path3 = Path.home() / "Prepayment_Analysis_Data" / "MORTGAGE30US.csv"
mort_rates_df = pd.read_csv(data_path3, sep=',', header=0)
mort_rates_df['mort_rate_date'] = pd.to_datetime(mort_rates_df['observation_date'])
mort_rates_df = mort_rates_df.drop(columns=['observation_date'])
mort_rates_df['MORTGAGE30US'] = mort_rates_df['MORTGAGE30US']
mort_rates_df = mort_rates_df[mort_rates_df['mort_rate_date'] > '2011-01-01'].copy()
mort_rates_df = mort_rates_df.sort_values('mort_rate_date').reset_index(drop=True)

#Add in immediately prior mortgage rate and calculate SATO
#Sort mortgage and rate dataframes by date
mort_clean_df = mort_clean_df.sort_values('ORIG_DATE1')
mort_rates_df = mort_rates_df.sort_values('mort_rate_date')

#Perform the "nearest" join
mort_clean_df = pd.merge_asof(
    mort_clean_df, 
    mort_rates_df[['mort_rate_date', 'MORTGAGE30US']], 
    left_on='ORIG_DATE1', 
    right_on='mort_rate_date', 
    direction='nearest'
)


mort_clean_df['SATO'] = mort_clean_df['ORIG_RATE'] - mort_clean_df['MORTGAGE30US']
mort_clean_df.drop(['mort_rate_date', 'MORTGAGE30US'], axis=1, inplace=True)

In [6]:
#Create prediction dataframe with unique LOAN_IDs and relevant features
pred_df_base= mort_clean_df[["LOAN_ID", "ORIG_RATE", "ORIG_DATE1", "OLTV", "CSCORE_MAX", "coop_condo_dummy", 
                             "SATO", "ORIG_UPB", "STATE", "MSA", 
                             "QTR_ORIG"]].drop_duplicates()
del mort_clean_df
gc.collect()
pred_df_base.shape

(192770, 11)

In [7]:
#Function to reduce memory usage of dataframe by downcasting numeric types and converting low-cardinality objects to categories
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype, is_categorical_dtype

def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        
        # Only process numeric columns (skip strings, categories, and datetimes)
        if is_numeric_dtype(col_type) and not isinstance(col_type, pd.CategoricalDtype):
            c_min = df[col].min()
            c_max = df[col].max()
            
            # Skip columns that are entirely NaN
            if pd.isna(c_min) or pd.isna(c_max):
                continue

            # Handling Integer types
            if 'int' in str(col_type).lower():
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            
            # Handling Float types
            else:
                # Use float32 as the safe minimum for financial calculations
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        
        # Convert objects or strings to category if they have low "cardinality"
        elif not is_datetime64_any_dtype(col_type):
            # Only convert to category if the number of unique values is < 50% of total
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype('category')

    if verbose:
        end_mem = df.memory_usage().sum() / 1024**2
        print(f'Memory usage decreased to {end_mem:5.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    
    return df

reduce_mem_usage(pred_df_base)

Memory usage decreased to  9.02 MB (38.7% reduction)


,LOAN_ID,ORIG_RATE,ORIG_DATE1,OLTV,CSCORE_MAX,coop_condo_dummy,SATO,ORIG_UPB,STATE,MSA,QTR_ORIG
0,000135825000,3.356,2020-12-01,57.0,769.0,0,0.646,509000.0,MI,19820,2020Q4
31,000135183288,6.250,2021-05-01,71.0,752.0,0,3.270,283000.0,FL,38940,2021Q2
32,000135236667,3.750,2021-05-01,45.0,809.0,0,0.770,214000.0,FL,34940,2021Q2
89,000135213478,4.250,2021-06-01,90.0,809.0,0,1.260,175000.0,FL,00000,2021Q2
98,000135191016,3.741,2021-06-01,65.0,766.0,0,0.751,647000.0,WY,00000,2021Q2
...,...,...,...,...,...,...,...,...,...,...,...
4984553,000135866355,6.200,2023-03-01,59.0,723.0,0,-0.450,349000.0,TX,19100,2023Q1
4984632,000135866373,6.875,2023-03-01,55.0,790.0,0,0.225,255000.0,ID,00000,2023Q1
4984633,000135848641,6.500,2023-03-01,64.0,801.0,1,-0.150,205000.0,MI,35660,2023Q1
4984637,000135866376,7.250,2023-03-01,95.0,812.0,0,0.600,770000.0,CA,31080,2023Q1


In [8]:
#GET unique origination date/rate combinations to build refi benefit and currentout tables
unique_combinations = pred_df_base[['ORIG_RATE', 'ORIG_DATE1']].drop_duplicates().reset_index(drop=True)

#unique_dates = unique_combinations[['ORIG_DATE1']].drop_duplicates().reset_index(drop=True)
#constuct monthly observations for each of these combinations up to Dec 2019
#for each unique date, create a date range from that date to Dec 2019

date_list = []
for i in range(len(unique_combinations)):
    date_df = pd.DataFrame({'DATE_NOW': pd.date_range(start=unique_combinations['ORIG_DATE1'][i], end='2024-12-01', freq='MS')})
    date_df['ORIG_DATE1']=unique_combinations['ORIG_DATE1'][i]
    date_df['ORIG_RATE']=unique_combinations['ORIG_RATE'][i]
    date_df['LOAN_AGE'] = ((date_df['DATE_NOW'].dt.year - date_df['ORIG_DATE1'].dt.year) * 12 + (date_df['DATE_NOW'].dt.month - date_df['ORIG_DATE1'].dt.month)).astype('int32')    
    date_list.append(date_df)

# Combine into single dataframe
dates_df = pd.concat(date_list, ignore_index=True)
dates_df.head()

,DATE_NOW,ORIG_DATE1,ORIG_RATE,LOAN_AGE
0,2020-12-01,2020-12-01,3.356,0
1,2021-01-01,2020-12-01,3.356,1
2,2021-02-01,2020-12-01,3.356,2
3,2021-03-01,2020-12-01,3.356,3
4,2021-04-01,2020-12-01,3.356,4


In [9]:
#calculate burnout variable
# Convert to numpy arrays for fast searching
rate_dates = mort_rates_df['mort_rate_date'].values
rate_values = mort_rates_df['MORTGAGE30US'].values

burn_df = dates_df[dates_df['LOAN_AGE'] >= 5].copy().reset_index()
burn_df['adjusted_end_date'] = burn_df['DATE_NOW'] - pd.DateOffset(months=3)


# Ultra-fast vectorized approach using binary search
def find_min_rate_fast(orig_dates, end_dates):
    """Find minimum rate in window using binary search on sorted dates"""
    min_rates = np.full(len(orig_dates), np.nan)
    min_dates = np.empty(len(orig_dates), dtype='datetime64[ns]')
    min_dates[:] = np.datetime64('NaT')
    
    for i in range(len(orig_dates)):
        # Binary search to find date range
        start_idx = bisect_right(rate_dates, orig_dates[i])
        end_idx = bisect_right(rate_dates, end_dates[i])
        
        if start_idx < end_idx:
            # Get rates in window
            window_rates = rate_values[start_idx:end_idx]
            min_idx_rel = np.argmin(window_rates)
            min_idx_abs = start_idx + min_idx_rel
            
            min_rates[i] = rate_values[min_idx_abs]
            min_dates[i] = rate_dates[min_idx_abs]
    
    return min_rates, min_dates

# Convert to numpy for speed
orig_dates_np = burn_df['ORIG_DATE1'].values.astype('datetime64[ns]')
end_dates_np = burn_df['adjusted_end_date'].values.astype('datetime64[ns]')

min_rates, min_dates = find_min_rate_fast(orig_dates_np, end_dates_np)

burn_df['min_past_mortgage_rate'] = min_rates
burn_df['min_rate_date'] = pd.to_datetime(min_dates)

burn_df['current_payment'] = nf.pmt(burn_df['ORIG_RATE']/1200, 360, 1000, fv=0, when='start')
burn_df['age_at_last_min'] = ((burn_df['min_rate_date'].dt.year - burn_df['ORIG_DATE1'].dt.year) * 12 + (burn_df['min_rate_date'].dt.month - burn_df['ORIG_DATE1'].dt.month))
burn_df['months_remaining_last_min'] = 360 - burn_df['age_at_last_min']
burn_df['upb_at_last_min'] = nf.pv(rate=burn_df['ORIG_RATE']/1200, nper=burn_df['months_remaining_last_min'], 
                                      pmt=burn_df['current_payment'])
burn_df['pv_at_last_min'] = nf.pv(rate=burn_df['min_past_mortgage_rate'].values/1200, nper=burn_df['months_remaining_last_min'].values, pmt=burn_df['current_payment'].values)
burn_df['refi_benefit_past'] = burn_df['upb_at_last_min'] - burn_df['pv_at_last_min']


burn_df.drop(['index', 'DATE_NOW', 'adjusted_end_date', 'min_past_mortgage_rate', 'min_rate_date', 'current_payment', 
             'age_at_last_min', 'months_remaining_last_min', 'upb_at_last_min', 'pv_at_last_min'], axis=1, inplace=True)

burn_df.head()

,ORIG_DATE1,ORIG_RATE,LOAN_AGE,refi_benefit_past
0,2020-12-01,3.356,5,-93.86486
1,2020-12-01,3.356,6,-93.86486
2,2020-12-01,3.356,7,-93.86486
3,2020-12-01,3.356,8,-93.86486
4,2020-12-01,3.356,9,-93.86486


In [10]:
#CURRENT REFI INCENTIVE INPUTS - MINIMUM INTEREST RATE IN 3 MONTHS PRIOR TO CURRENT DATE
#Get current refi benefit

current_df = dates_df[dates_df['LOAN_AGE'] >= 5].copy().reset_index()
current_df['recent_beginning'] = current_df['DATE_NOW'] - pd.DateOffset(months=3)

# Convert to numpy for speed
start_dates_np = current_df['recent_beginning'].values.astype('datetime64[ns]')
now_dates_np = current_df['DATE_NOW'].values.astype('datetime64[ns]')

min_rates, min_dates = find_min_rate_fast(start_dates_np, now_dates_np)

current_df['min_past_mortgage_rate'] = min_rates
current_df['min_rate_date'] = pd.to_datetime(min_dates)

current_df['current_payment'] = nf.pmt(current_df['ORIG_RATE']/1200, 360, 1000, fv=0, when='start')
current_df['age_at_last_min'] = ((current_df['min_rate_date'].dt.year - current_df['ORIG_DATE1'].dt.year) * 12 + (current_df['min_rate_date'].dt.month - current_df['ORIG_DATE1'].dt.month))
current_df['months_remaining_last_min'] = 360 - current_df['age_at_last_min']
current_df['upb_at_last_min'] = nf.pv(rate=current_df['ORIG_RATE']/1200, nper=current_df['months_remaining_last_min'], 
                                      pmt=current_df['current_payment'])
current_df['pv_at_last_min'] = nf.pv(rate=current_df['min_past_mortgage_rate'].values/1200, nper=current_df['months_remaining_last_min'].values, pmt=current_df['current_payment'].values)
current_df['refi_benefit_now'] = current_df['upb_at_last_min'] - current_df['pv_at_last_min']
current_df['UPB_MULT']=nf.pv(rate=current_df['ORIG_RATE']/1200, nper=360-current_df['LOAN_AGE'], 
                                      pmt=current_df['current_payment'])/1000

current_df.drop(['index','recent_beginning', 'min_past_mortgage_rate', 'min_rate_date', 'current_payment', 
             'age_at_last_min', 'months_remaining_last_min', 'upb_at_last_min', 'pv_at_last_min' ], axis=1, inplace=True)
current_df.head()


,DATE_NOW,ORIG_DATE1,ORIG_RATE,LOAN_AGE,refi_benefit_now,UPB_MULT
0,2021-05-01,2020-12-01,3.356,5,-82.271925,0.989120
1,2021-06-01,2020-12-01,3.356,6,-52.972577,0.987488
2,2021-07-01,2020-12-01,3.356,7,-54.077332,0.985851
3,2021-08-01,2020-12-01,3.356,8,-73.834831,0.984210
4,2021-09-01,2020-12-01,3.356,9,-74.878229,0.982564


In [11]:
#Get and clean HPI DATA at state and MSA levels
KEEP_COLUMNS1= ['level','place_id','yr',
                'period', 'index_sa']

hpi_df = pd.read_csv("~/Prepayment_Analysis_Data/hpi_master.csv")
hpi_df1 = hpi_df[(hpi_df['hpi_type'] == 'traditional') & (hpi_df['hpi_flavor'] == 'purchase-only')
                  & (hpi_df['level'].isin(['MSA', 'State']) & (hpi_df['yr']>2019))]
hpi_df1=hpi_df1[KEEP_COLUMNS1]
hpi_df1 = hpi_df1.reset_index(drop=True)
#keep only msa-level hpi
hpi_msas = hpi_df1[hpi_df1['level']=='MSA'].copy()
hpi_msas.rename(columns={'index_sa':'msa_index', 'place_id':'metdiv10', 'place_name':'msa_name'}, inplace=True)
hpi_msas['metdiv10'] = hpi_msas['metdiv10'].astype(str)
#The FHFA HPI data is at the metropolitan division level for some MSAs while that is not the case
# for the Fannie data, so we need to aggregate to MSA level for those MSAs
#load crosswalk file - metropolitan division to msa
xwalk_df = pd.read_csv("~/Prepayment_Analysis_Data/msa_md_xwalk_2020.csv")
xwalk_df = pd.DataFrame(xwalk_df.reset_index())
xwalk_df.drop(['2010 CBSA name', '2010 Metro div name', 'Total housing units (2010)'], axis=1, inplace=True)
xwalk_df.rename(columns={'cbsa10 to metdiv10 allocation factor': 'weight1'}, inplace=True)
xwalk_df['metdiv10'] = xwalk_df['metdiv10'].astype(str)
xwalk_df['MSA'] = xwalk_df['cbsa10'].astype(str)
xwalk_df.dtypes
#get data for cbsas that need to be calculated from metdivs
missing_msas = hpi_msas.merge(xwalk_df, how='inner', on='metdiv10')
#calculate weighted average hpi for those msas
weighted_hpi_df = missing_msas.groupby(['MSA', 'yr', 'period'], as_index=False).apply(
    lambda x: np.average(x['msa_index'], weights=x['weight1']))
weighted_hpi_df.rename(columns={None:'msa_index'}, inplace=True)

hpi_msas.rename(columns={'metdiv10':'MSA'}, inplace=True)
#combine weighted hpi msas with regular hpi msas
hpi_msas1 = pd.concat([hpi_msas, weighted_hpi_df], ignore_index=True)
hpi_msas1['year_quarter'] = pd.PeriodIndex(year=hpi_msas1['yr'], quarter=hpi_msas1['period'], freq='Q')
hpi_msas1.drop(['level', 'yr', 'period'], axis=1, inplace=True)

hpi_msas1.head()
# #state HPI
hpi_state = hpi_df1[hpi_df1['level']=='State'].copy()
hpi_state['year_quarter'] = pd.PeriodIndex(year=hpi_state['yr'], quarter=hpi_state['period'], freq='Q')
hpi_state.drop(['level', 'yr', 'period'], axis=1, inplace=True)
hpi_state.rename(columns={'index_sa':'state_index', 
                          'place_id':'STATE'}, inplace=True)
hpi_state.head()


/var/folders/rk/xd9hdkfd1bz1rk65mnhlmn7c0000gn/T/ipykernel_3347/2789049188.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_hpi_df = missing_msas.groupby(['MSA', 'yr', 'period'], as_index=False).apply(
/var/folders/rk/xd9hdkfd1bz1rk65mnhlmn7c0000gn/T/ipykernel_3347/2789049188.py:34: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  hpi_msas1['year_quarter'] = pd.PeriodIndex(year=hpi_msas1['yr'], quarter=hpi_msas1['period'], freq='Q')
/var/folders/rk/xd9hdkfd1bz1rk65mnhlmn7c0000gn/T/ipykernel_3347/2789049188.py:40: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  hpi_state['year

,STATE,state_index,year_quarter
2300,AK,267.26,2020Q1
2301,AK,272.21,2020Q2
2302,AK,273.57,2020Q3
2303,AK,287.79,2020Q4
2304,AK,290.91,2021Q1


In [12]:
#add home price appreciation

#GET unique origination date/location combinations to build hpa tables
unique_locations = pred_df_base[['QTR_ORIG', 'STATE', 'MSA']].drop_duplicates().reset_index(drop=True)

#construct quarterly observations for each of these combinations up to Q4 2019

quarter_list = []
for i in range(len(unique_locations)):  
    date_df = pd.DataFrame({'QTR_NOW': pd.date_range(start=unique_locations['QTR_ORIG'][i].to_timestamp(), end='2024-10-01', freq='QS')})  # Changed end date format
    date_df['QTR_NOW'] = date_df['QTR_NOW'].dt.to_period('Q')
    date_df['QTR_ORIG']=unique_locations['QTR_ORIG'][i]
    date_df['STATE']=unique_locations['STATE'][i]  
    date_df['MSA']=unique_locations['MSA'][i]
    quarter_list.append(date_df)

quarter_df = pd.concat(quarter_list, ignore_index=True)  
quarter_df.tail()
#merge hpi data into unique_locations data
quarter_df = quarter_df.merge(hpi_msas1, how='left', left_on=['MSA','QTR_NOW'],
                                    right_on=['MSA','year_quarter'])
quarter_df.drop(['year_quarter'], axis=1, inplace=True)
quarter_df.rename(columns={'msa_index':'hpi_msa_now', 'place_id':'msa_place_id'}, inplace=True)
quarter_df = quarter_df.merge(hpi_msas1, how='left', left_on=['MSA', 'QTR_ORIG'],
                                    right_on=['MSA', 'year_quarter'])
quarter_df.drop(['year_quarter'], axis=1, inplace=True)
quarter_df.rename(columns={'msa_index':'hpi_msa_orig'}, inplace=True)
quarter_df['hpi_growth_msa'] = (quarter_df['hpi_msa_now'] - quarter_df['hpi_msa_orig']) / quarter_df['hpi_msa_orig']
quarter_df.drop(['hpi_msa_orig', 'hpi_msa_now'], axis=1, inplace=True)
quarter_df.head()

quarter_df = quarter_df.merge(hpi_state, how='left', left_on=['STATE','QTR_NOW'],
                                   right_on=['STATE','year_quarter'])
quarter_df.drop(['year_quarter'], axis=1, inplace=True)
quarter_df.rename(columns={'state_index':'hpi_state_now'}, inplace=True)
quarter_df = quarter_df.merge(hpi_state, how='left', left_on=['STATE', 'QTR_ORIG'],
                                   right_on=['STATE', 'year_quarter'])
quarter_df.drop(['year_quarter'], axis=1, inplace=True)
quarter_df.rename(columns={'state_index':'hpi_state_orig'}, inplace=True)
quarter_df['hpi_growth_state'] = (quarter_df['hpi_state_now'] - quarter_df['hpi_state_orig']) / quarter_df['hpi_state_orig']
quarter_df.drop(['hpi_state_orig', 'hpi_state_now'], axis=1, inplace=True)
quarter_df.describe()
#want to create single hpi growth variables - fill in state when msa is missing
quarter_df['hpi_growth'] = quarter_df['hpi_growth_msa'].fillna(quarter_df['hpi_growth_state'])
quarter_df.drop(['hpi_growth_msa', 'hpi_growth_state'], axis=1, inplace=True)
reduce_mem_usage(quarter_df)
quarter_df.head()

Memory usage decreased to  0.15 MB (75.1% reduction)


,QTR_NOW,QTR_ORIG,STATE,MSA,hpi_growth
0,2020Q4,2020Q4,MI,19820,0.000000
1,2021Q1,2020Q4,MI,19820,0.035110
2,2021Q2,2020Q4,MI,19820,0.062098
3,2021Q3,2020Q4,MI,19820,0.095119
4,2021Q4,2020Q4,MI,19820,0.122983


In [ ]:
#merge pred_df=pred_df_base.merge(quarter_df, how='left',on=['QTR_ORIG','STATE', 'MSA'])
pred_df.drop(columns=['QTR_ORIG','STATE','MSA'], inplace=True)
pred_df.head()

,LOAN_ID,ORIG_RATE,ORIG_DATE1,OLTV,CSCORE_MAX,coop_condo_dummy,SATO,ORIG_UPB,QTR_NOW,hpi_growth
0,000135825000,3.356,2020-12-01,57.0,769.0,0,0.646,509000.0,2020Q4,0.000000
1,000135825000,3.356,2020-12-01,57.0,769.0,0,0.646,509000.0,2021Q1,0.035110
2,000135825000,3.356,2020-12-01,57.0,769.0,0,0.646,509000.0,2021Q2,0.062098
3,000135825000,3.356,2020-12-01,57.0,769.0,0,0.646,509000.0,2021Q3,0.095119
4,000135825000,3.356,2020-12-01,57.0,769.0,0,0.646,509000.0,2021Q4,0.122983


In [ ]:
#merge burnout data
burn_df = burn_df.merge(current_df, how='outer', on=['ORIG_RATE', 'ORIG_DATE1', 'LOAN_AGE'])
burn_df['QTR_NOW'] = burn_df['DATE_NOW'].dt.to_period('Q')
reduce_mem_usage(burn_df)
burn_df.head()

Memory usage decreased to  2.30 MB (39.3% reduction)


,ORIG_DATE1,ORIG_RATE,LOAN_AGE,refi_benefit_past,DATE_NOW,refi_benefit_now,UPB_MULT,QTR_NOW
0,2022-11-01,2.75,5,336.430847,2023-04-01,321.481110,0.988739,2023Q2
1,2022-11-01,2.75,6,325.456116,2023-05-01,321.481110,0.986932,2023Q2
2,2022-11-01,2.75,7,321.481110,2023-06-01,331.771729,0.985121,2023Q2
3,2022-11-01,2.75,8,321.481110,2023-07-01,331.771729,0.983305,2023Q3
4,2022-11-01,2.75,9,321.481110,2023-08-01,336.050873,0.981486,2023Q3


In [ ]:
#Final merges and calcualtions
pred_df = pred_df.merge(burn_df, how='inner', on=['ORIG_RATE', 'ORIG_DATE1', 'QTR_NOW'])   
pred_df['refi_benefit_past'] = pred_df['refi_benefit_past']*pred_df['ORIG_UPB'] / 1000
pred_df['refi_benefit_now'] = pred_df['refi_benefit_now']*pred_df['ORIG_UPB'] / 1000
pred_df.drop('QTR_NOW', axis=1, inplace=True)
pred_df=pred_df.merge(actual_prepay_df, how='left', on=['LOAN_ID', 'LOAN_AGE'])
pred_df['PREPAY_FLAG'] = pred_df['PREPAY_FLAG'].fillna(0)
pred_df.describe()

,ORIG_RATE,ORIG_DATE1,OLTV,CSCORE_MAX,coop_condo_dummy,SATO,ORIG_UPB,hpi_growth,LOAN_AGE,refi_benefit_past,DATE_NOW,refi_benefit_now,UPB_MULT,PREPAY_FLAG
count,3.673094e+06,3673094,3.673094e+06,3.673094e+06,3.673094e+06,3.673094e+06,3.673094e+06,3.673094e+06,3.673094e+06,3.673094e+06,3673094,3.673094e+06,3.673094e+06,3.673094e+06
mean,6.371447e+00,2022-12-26 18:46:59.501174784,7.666913e+01,7.617407e+02,9.947880e-02,-2.484201e-02,3.222919e+05,6.974871e-02,1.408204e+01,-8.578972e+03,2024-02-28 01:44:14.254318848,6.160236e+03,9.810988e-01,5.773607e-03
min,2.750000e+00,2020-12-01 00:00:00,3.000000e+00,6.200000e+02,0.000000e+00,-4.200000e+00,1.500000e+04,-1.049948e-01,5.000000e+00,-3.462239e+05,2021-05-01 00:00:00,-3.317906e+05,9.146638e-01,0.000000e+00
25%,5.990000e+00,2022-12-01 00:00:00,6.900000e+01,7.370000e+02,0.000000e+00,-4.200000e-01,1.910000e+05,4.191205e-02,9.000000e+00,-1.990740e+04,2023-10-01 00:00:00,-7.695506e+03,9.766697e-01,0.000000e+00
50%,6.375000e+00,2023-01-01 00:00:00,8.000000e+01,7.700000e+02,0.000000e+00,-1.500010e-02,2.900000e+05,6.510466e-02,1.400000e+01,-7.142410e+03,2024-03-01 00:00:00,3.954753e+03,9.814187e-01,0.000000e+00
75%,6.875000e+00,2023-02-01 00:00:00,9.300000e+01,7.940000e+02,0.000000e+00,3.999998e-01,4.200000e+05,9.363018e-02,1.900000e+01,3.687752e+03,2024-08-01 00:00:00,1.752431e+04,9.859320e-01,0.000000e+00
max,8.625000e+00,2023-03-01 00:00:00,9.700000e+01,8.340000e+02,1.000000e+00,4.620000e+00,2.095000e+06,5.105370e-01,4.800000e+01,5.061901e+05,2024-12-01 00:00:00,6.212189e+05,9.900787e-01,1.000000e+00
std,6.699963e-01,NaN,1.851781e+01,3.976315e+01,2.993039e-01,6.512860e-01,1.745556e+05,4.000832e-02,5.598529e+00,2.614584e+04,NaN,2.705091e+04,5.821970e-03,7.576459e-02


In [ ]:
#This is the final out-of-sample prediction data set
pred_df.to_csv('~/Prepayment_Analysis_Data/prediction_df_oos.csv', index=False, header=True)